# ブートストラッピングの意味ドリフトを観察して抑える

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/20_lexical_semantics/semantic_drift.ipynb)

NLP クックブック 20-04 ／ 語の意味と意味の変化

ブートストラッピングを繰り返すと、シードと関係のない語ばかりが集まるようになってしまう（意味ドリフト）。なぜ起きるかを確かめ、抑える方法を試したい。

In [1]:
!pip install -q datasets fugashi unidic-lite

In [2]:
import itertools, math, re
from collections import Counter
import numpy as np
import scipy.sparse as sp
from datasets import load_dataset
from fugashi import Tagger

ds = load_dataset("wikimedia/wikipedia", "20231101.ja", split="train", streaming=True)
articles = [ex["text"][:1500] for ex in itertools.islice(ds, 12000)]
tagger = Tagger()

cooc = Counter()
for a in articles:
    for s in re.split(r"(?<=。)|\n", a):
        if not 10 <= len(s) <= 150:
            continue
        c = []
        for w in tagger(s):
            if w.feature.pos1 == "名詞" and c and c[-1][1]:
                c[-1] = (c[-1][0] + w.surface, True)
            else:
                c.append((w.surface, w.feature.pos1 == "名詞"))
        c = [("<s>", False)] + c + [("</s>", False), ("</s>", False)]
        for i in range(1, len(c) - 2):
            if c[i][1] and len(c[i][0]) >= 2 and not c[i][0].isdigit():
                cooc[c[i][0], f"{c[i - 1][0]} X {c[i + 1][0]}"] += 1
                cooc[c[i][0], f"X {c[i + 1][0]} {c[i + 2][0]}"] += 1

n_inst, n_pat = Counter(), Counter()
for (i, p), n in cooc.items():
    n_inst[i] += n; n_pat[p] += n
instances = [i for i, n in n_inst.items() if n >= 5]
patterns = [p for p, n in n_pat.items() if n >= 5]
inst_id, pat_id = {w: k for k, w in enumerate(instances)}, {w: k for k, w in enumerate(patterns)}
total, rows, cols, vals = sum(cooc.values()), [], [], []
for (i, p), n in cooc.items():
    if i in inst_id and p in pat_id and (pmi := math.log2(n * total / (n_inst[i] * n_pat[p]))) > 0:
        rows.append(pat_id[p]); cols.append(inst_id[i]); vals.append(pmi)
M = sp.csr_matrix((vals, (rows, cols)), shape=(len(patterns), len(instances)))
M = sp.csr_matrix(M / M.max())

def seed_vector(seeds):
    v = np.zeros(len(instances)); v[[inst_id[s] for s in seeds]] = 1.0
    return v

def top(scores, k=12):
    return " ".join(instances[i] for i in np.argsort(-scores)[:k])

In [3]:
# 反復を増やすと、どのシードから始めても同じ語に収束する
SEEDS = {"国": ["日本", "フランス", "中国"], "楽器": ["ピアノ", "ギター", "ヴァイオリン"], "作家": ["夏目漱石", "森鴎外", "芥川龍之介"]}
final = {}
for name, seeds in SEEDS.items():
    v = seed_vector(seeds)
    for it in range(1, 31):
        p = M @ v; p /= p.max(); v = M.T @ p; v /= v.max()
        if it in (1, 2, 5, 30):
            print(f"{name} 反復 {it:2}: {top(v)}")
    final[name] = v
print("30 回後の順位の一致 (上位 100 語の重なり):", len(set(np.argsort(-final["国"])[:100]) & set(np.argsort(-final["楽器"])[:100])))

国 反復  1: 日本 フランス 中国 イギリス アメリカ ドイツ イタリア ヨーロッパ ロシア スペイン オランダ インド
国 反復  2: 日本 フランス イギリス アメリカ 中国 ドイツ ヨーロッパ イタリア ロシア 多く スペイン インド
国 反復  5: もの 作品 地域 こと コンピュータ 言語 人間 問題 概念 データ 状態 ゲーム
国 反復 30: もの 作品 地域 こと コンピュータ 言語 人間 問題 概念 データ 状態 ゲーム
楽器 反復  1: ピアノ ヴァイオリン ギター 音楽 オルガン ボーカル キーボード ベース ホルン ミカン ロック アブラナ
楽器 反復  2: もの 作品 音楽 コンピュータ 地域 ピアノ 人間 データ ゲーム 言語 システム こと
楽器 反復  5: もの 作品 地域 こと コンピュータ 言語 人間 問題 概念 データ 状態 ゲーム
楽器 反復 30: もの 作品 地域 こと コンピュータ 言語 人間 問題 概念 データ 状態 ゲーム
作家 反復  1: 芥川龍之介 夏目漱石 セルバンテス 川端康成 太宰治 アシモフ 三島由紀夫 本居宣長 宮沢賢治 ヘーシオドス ツヴァイク ヘロドトス
作家 反復  2: イギリス フランス 日本 アメリカ 人間 コンピュータ ドイツ 多く 中国 作品 自分 最初
作家 反復  5: もの 作品 地域 こと コンピュータ 人間 言語 問題 概念 データ 状態 ゲーム
作家 反復 30: もの 作品 地域 こと コンピュータ 言語 人間 問題 概念 データ 状態 ゲーム
30 回後の順位の一致 (上位 100 語の重なり): 100


In [4]:
# 理由: この反復は行列 A = MᵀM を繰り返し掛けること (べき乗法) と同じで、シードに関係なく A の第 1 固有ベクトルに収束する
from scipy.sparse.linalg import svds

_, s, vt = svds(M, k=1)
principal = np.abs(vt[0])
print("第 1 固有ベクトルの上位:", top(principal))
print("30 回後のスコアとの相関:", round(float(np.corrcoef(principal, final["国"])[0, 1]), 4))
print("収束先に多い語は、多くのパターンと共起する語:", [(instances[i], int((M[:, i] > 0).sum())) for i in np.argsort(-principal)[:6]])

第 1 固有ベクトルの上位: もの 作品 地域 こと コンピュータ 言語 人間 問題 概念 データ 状態 ゲーム
30 回後のスコアとの相関: 1.0
収束先に多い語は、多くのパターンと共起する語: [('もの', 2574), ('作品', 1147), ('地域', 968), ('こと', 3448), ('コンピュータ', 631), ('言語', 811)]


In [5]:
# 対策 1: フォン・ノイマン カーネル。反復 1 回目、2 回目、... の結果を、回数が増えるほど小さい重み (α の n 乗) で足し合わせる
lam = float(s[0]) ** 2                              # A の最大固有値。α / lam を掛けることで和が収束する

def von_neumann(seeds, alpha, terms=50):
    term, score = seed_vector(seeds), np.zeros(len(instances))
    for _ in range(terms):
        term = (alpha / lam) * (M.T @ (M @ term)); score += term
    return score

# 対策 2: 正則化ラプラシアン カーネル。多くの語とつながっている語 (ハブ) のスコアを、次数で割り引く
from scipy.sparse.linalg import LinearOperator, cg

degree = np.asarray(M.T @ (M @ np.ones(len(instances)))).ravel()
d = 1 / np.sqrt(degree + 1e-12)

def regularized_laplacian(seeds, beta):
    op = LinearOperator((len(instances),) * 2, matvec=lambda x: x + beta * (x - d * (M.T @ (M @ (d * x)))))    # (I + βL) x = シード を解く
    return cg(op, seed_vector(seeds), maxiter=200)[0]

for name, seeds in SEEDS.items():
    print(name)
    for alpha in [0.01, 0.9, 0.99]:
        print(f"   フォン・ノイマン α={alpha}: {top(von_neumann(seeds, alpha))}")
    for beta in [1, 100]:
        print(f"   正則化ラプラシアン β={beta}: {top(regularized_laplacian(seeds, beta))}")

国
   フォン・ノイマン α=0.01: 日本 フランス 中国 イギリス アメリカ ドイツ イタリア ヨーロッパ ロシア スペイン オランダ インド
   フォン・ノイマン α=0.9: 日本 フランス 中国 イギリス アメリカ ドイツ ヨーロッパ イタリア ロシア 多く 地域 スペイン
   フォン・ノイマン α=0.99: 日本 フランス 中国 もの イギリス アメリカ 地域 作品 ドイツ こと ヨーロッパ 人間
   正則化ラプラシアン β=1: 日本 フランス 中国 イギリス アメリカ ドイツ 東シナ カスピ イタリア アドリア マルマラ エーゲ
   正則化ラプラシアン β=100: 日本 フランス 中国 イギリス アメリカ ドイツ ヨーロッパ ロシア 地域 イタリア もの コンピュータ
楽器
   フォン・ノイマン α=0.01: ピアノ ヴァイオリン ギター 音楽 オルガン ボーカル キーボード ベース ホルン ミカン ロック アブラナ
   フォン・ノイマン α=0.9: ピアノ ヴァイオリン ギター もの 作品 音楽 地域 コンピュータ こと データ 言語 人間
   フォン・ノイマン α=0.99: もの 作品 地域 こと ピアノ コンピュータ 言語 人間 問題 データ 概念 音楽
   正則化ラプラシアン β=1: ピアノ ヴァイオリン ギター Tuber オランダイチゴ ヴィオール アブラナ バイソン ハンニチバナ ミカン 交直 東京大学理学部情報科学
   正則化ラプラシアン β=100: ピアノ ギター ヴァイオリン もの コンピュータ 作品 人間 地域 音楽 データ ゲーム システム
作家
   フォン・ノイマン α=0.01: 芥川龍之介 夏目漱石 セルバンテス 川端康成 太宰治 アシモフ 三島由紀夫 本居宣長 宮沢賢治 ヘーシオドス ツヴァイク ヘロドトス
   フォン・ノイマン α=0.9: 芥川龍之介 夏目漱石 もの 作品 地域 人間 コンピュータ ゲーム こと イギリス 日本 データ
   フォン・ノイマン α=0.99: もの 作品 地域 こと コンピュータ 人間 言語 データ 問題 概念 ゲーム 状態
   正則化ラプラシアン β=1: 芥川龍之介 夏目漱石 森鴎外 グロティウス セルバンテス 辻村深月 ミシュレ 山県大

- 意味ドリフトは、多くのパターンと共起する一般的な語（「もの」「作品」「こと」）が一度入ると、そこから無関係なパターンと語が芋づる式に増えていく現象。
- 簡略化した Espresso の反復は、行列のべき乗法と同じ計算になっている。繰り返すほどシードの情報が消え、どのシードでも同じ順位（グラフの中心性）に収束する。これは HITS アルゴリズムが権威ノードを求める計算と同じ。
- フォン・ノイマン カーネルは、α が小さいとシードの近くを重視し、大きいとグラフ全体の中心性に近づく。正則化ラプラシアンは、ハブを通る経路を割り引くので、一般的な語に流れにくい（その代わり、頻度の低い語が上位に来やすい）。
- 実際の Espresso は、各反復で信頼度の高いパターンと語だけを少数ずつ残すことで、ドリフトを遅らせている。反復回数を増やしすぎない、一般的すぎるパターンをストップリストで除く、といった対策も使われる。
- どの方法でも、パラメータ（α、β、反復回数）で「シードへの近さ」と「広がり」の釣り合いを調整することになる。少数の正解データで確かめながら決める。
- 少数の正解をグラフの辺に沿って広げる考え方は、文書の分類にも使える（テキスト分類の章の「少数のラベルをグラフで広げて分類する（scikit-learn の LabelSpreading）」）。ハブを通って違うものに広がってしまう問題も共通している。